# 9.5 Batch Prediction

本 notebook 示範完整批次推論流程：訓練模型、保存 scaler、建立待預測 CSV、載入模型、批次推論並輸出 prediction CSV。


## 1. 載入套件


In [ ]:
import json
import os
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(tf.__version__)


## 2. 建立訓練資料


In [ ]:
X, y = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=7,
    n_redundant=2,
    n_classes=3,
    n_clusters_per_class=1,
    class_sep=1.2,
    flip_y=0.03,
    random_state=SEED,
)
X = X.astype('float32')
y = y.astype('int64')
class_names = ['bronze', 'silver', 'gold']
feature_columns = [f'feature_{i}' for i in range(X.shape[1])]

x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
x_train, x_valid, y_train, y_valid = train_test_split(
    x_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=SEED
)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train).astype('float32')
x_valid_scaled = scaler.transform(x_valid).astype('float32')
x_test_scaled = scaler.transform(x_test).astype('float32')

print(x_train_scaled.shape, x_valid_scaled.shape, x_test_scaled.shape)


## 3. 訓練並保存模型


In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(SEED)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(feature_columns),), name='features'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(len(class_names), activation='softmax', name='probabilities'),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.fit(
    x_train_scaled,
    y_train,
    validation_data=(x_valid_scaled, y_valid),
    epochs=24,
    batch_size=32,
    verbose=0,
)
print(dict(zip(model.metrics_names, model.evaluate(x_test_scaled, y_test, verbose=0))))

artifact_dir = Path('artifacts/9_5_batch_prediction')
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / 'classifier.keras'
scaler_path = artifact_dir / 'scaler.joblib'
metadata_path = artifact_dir / 'metadata.json'

model.save(model_path)
joblib.dump(scaler, scaler_path)
metadata = {
    'class_names': class_names,
    'feature_columns': feature_columns,
    'model_version': 'demo-1.0.0',
}
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
print(model_path)
print(scaler_path)
print(metadata_path)


## 4. 建立待預測 CSV


In [ ]:
raw_batch = pd.DataFrame(x_test[:12], columns=feature_columns)
raw_batch.insert(0, 'sample_id', [f'sample_{i:03d}' for i in range(len(raw_batch))])
input_csv_path = artifact_dir / 'input_batch.csv'
raw_batch.to_csv(input_csv_path, index=False)
raw_batch.head()


## 5. 建立批次推論函式


In [ ]:
def run_batch_prediction(input_csv_path, output_csv_path, model_path, scaler_path, metadata_path):
    loaded_model = tf.keras.models.load_model(model_path)
    loaded_scaler = joblib.load(scaler_path)
    loaded_metadata = json.loads(Path(metadata_path).read_text(encoding='utf-8'))

    input_df = pd.read_csv(input_csv_path)
    feature_columns = loaded_metadata['feature_columns']
    missing_columns = sorted(set(feature_columns) - set(input_df.columns))
    if missing_columns:
        raise ValueError(f'Missing feature columns: {missing_columns}')

    features = input_df[feature_columns].to_numpy(dtype='float32')
    features_scaled = loaded_scaler.transform(features).astype('float32')
    probabilities = loaded_model.predict(features_scaled, batch_size=32, verbose=0)
    predicted_class = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    class_names = loaded_metadata['class_names']

    output_df = input_df[['sample_id']].copy() if 'sample_id' in input_df.columns else pd.DataFrame(index=input_df.index)
    output_df['predicted_class'] = predicted_class
    output_df['predicted_label'] = [class_names[i] for i in predicted_class]
    output_df['confidence'] = confidence
    output_df['model_version'] = loaded_metadata.get('model_version', 'unknown')

    for i, class_name in enumerate(class_names):
        output_df[f'prob_{class_name}'] = probabilities[:, i]

    output_df.to_csv(output_csv_path, index=False)
    return output_df

output_csv_path = artifact_dir / 'predictions.csv'
predictions = run_batch_prediction(
    input_csv_path=input_csv_path,
    output_csv_path=output_csv_path,
    model_path=model_path,
    scaler_path=scaler_path,
    metadata_path=metadata_path,
)
predictions.head()


## 6. 檢查輸出檔


In [ ]:
print(output_csv_path)
loaded_predictions = pd.read_csv(output_csv_path)
print(loaded_predictions.head())
print('rows:', len(loaded_predictions))


## 7. 大量資料的分批推論寫法


In [ ]:
def iter_csv_chunks(csv_path, chunksize=5):
    for chunk in pd.read_csv(csv_path, chunksize=chunksize):
        yield chunk

chunk_sizes = [len(chunk) for chunk in iter_csv_chunks(input_csv_path, chunksize=5)]
chunk_sizes


## 8. 套用自己的資料

請固定輸入欄位、前處理物件、模型版本與輸出欄位。正式批次任務建議記錄輸入檔名、輸出檔名、資料筆數、模型版本與執行時間，方便追蹤與回溯。
